# NPS Preditivo - Passo 7: Motor de Inferência e Fila de Risco
Este notebook demonstra de forma interativa e visual como o nosso modelo de produção escoreia novos dados operacionais recebidos, calcula a probabilidade individual de detração, segmenta os clientes em faixas de risco proativas e ordena uma fila priorizada para a tomada de decisões imediatas das áreas de logística e suporte.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Configurações estéticas de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False

## 1. Mapeamento de Caminhos e Carga dos Dados

In [ ]:
# Caminho absoluto dinâmico para rodar a partir de qualquer pasta
BASE_DIR = Path(os.getcwd()).resolve()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

INPUT_DATA_PATH = BASE_DIR / "data" / "raw" / "desafio_nps_fase_1.csv"
MODEL_PATH = BASE_DIR / "models" / "detractor_classifier.joblib"

# Fallback para ambientes temporários de teste
if not INPUT_DATA_PATH.exists():
    INPUT_DATA_PATH = Path("desafio_nps_fase_1.csv")
if not MODEL_PATH.exists():
    MODEL_PATH = Path("detractor_classifier.joblib")

df_new = pd.read_csv(INPUT_DATA_PATH)
print(f"✓ Dados operacionais carregados: {df_new.shape[0]} registros.")
df_new.head(3)

## 2. Carregar o Pipeline Preditivo Serializado

In [ ]:
print(f"Carregando pipeline de modelagem serializado de: {MODEL_PATH}")
pipeline = joblib.load(MODEL_PATH)
print("✓ Pipeline preditivo carregado com sucesso!")
print("Estrutura do Pipeline:")
pipeline

## 3. Inferência: Calcular Probabilidade de Detração
O pipeline pré-processa os novos dados operacionais (imputação e padronização das numéricas + One-Hot Encoding das categóricas) e executa o estimador cego a vazamento de dados.

In [ ]:
# Gerar previsões de probabilidade para a classe positiva (1 - Detrator)
probabilities = pipeline.predict_proba(df_new)
detractor_probs = probabilities[:, 1]

# Acoplar a probabilidade calculada ao DataFrame de novos pedidos
df_new["detractor_probability"] = detractor_probs
print("✓ Probabilidades preditas com sucesso!")

## 4. Classificação das Faixas de Risco (Risk Bands)
De acordo com os limites definidos pelo negócio:
- **Baixo Risco:** Probabilidade <= 50%
- **Alto Risco:** Probabilidade entre 50% e 75%
- **Risco Crítico:** Probabilidade > 75%

In [ ]:
def categorize_risk_band(probability):
    if probability <= 0.50:
        return "Baixo Risco"
    elif 0.50 < probability <= 0.75:
        return "Alto Risco"
    else:
        return "Risco Crítico"

df_new["risk_band"] = df_new["detractor_probability"].apply(categorize_risk_band)

risk_counts = df_new["risk_band"].value_counts()
risk_pcts = df_new["risk_band"].value_counts(normalize=True) * 100

print("Distribuição das Faixas de Risco Operacional:")
for band in risk_counts.index:
    print(f"- {band:15s}: {risk_counts[band]:5d} ({risk_pcts[band]:.2f}%)")

## 5. Visualização das Faixas de Risco

In [ ]:
plt.figure(figsize=(10, 6))
colors = {"Risco Crítico": "#d9534f", "Alto Risco": "#f0ad4e", "Baixo Risco": "#5cb85c"}
ax = sns.countplot(
    x="risk_band", 
    data=df_new, 
    palette=colors, 
    order=["Risco Crítico", "Alto Risco", "Baixo Risco"]
)

plt.title("Distribuição de Pedidos pelas Faixas de Risco de Detração", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Faixa de Risco", fontsize=12)
plt.ylabel("Quantidade de Pedidos", fontsize=12)

# Exibir os valores em cima de cada barra
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{height}\n({height/len(df_new)*100:.1f}%)',
                xy=(p.get_x() + p.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Priorização e Estruturação da Fila de Risco
Para apoiar a tomada de decisão da operação, a fila deve ser ordenada de modo que os clientes sob **Risco Crítico** com maior probabilidade fiquem no topo da fila, facilitando a ação ativa do suporte/logística.

In [ ]:
# Ordenar por prioridade (Risco Crítico primeiro, Alto Risco depois, e maior probabilidade)
risk_order = {"Risco Crítico": 0, "Alto Risco": 1, "Baixo Risco": 2}
df_new["risk_priority"] = df_new["risk_band"].map(risk_order)

df_prioritized = df_new.sort_values(
    by=["risk_priority", "detractor_probability"], 
    ascending=[True, False]
).drop(columns=["risk_priority"])

print("=== TOP 10 CLIENTES DE MAIOR RISCO (FILA DE PRIORIDADE CRÍTICA) ===")
cols_to_show = [
    "customer_id", "customer_region", "delivery_delay_days", 
    "customer_service_contacts", "complaints_count", "detractor_probability", "risk_band"
]
df_prioritized[cols_to_show].head(10)

## 7. Exportar a Base Escorada
Salvando a fila priorizada no arquivo definitivo de produção `scored_orders.csv` para consumo das APIs ou painéis operacionais.

In [ ]:
OUTPUT_FILE = BASE_DIR / "data" / "processed" / "scored_orders.csv"
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df_prioritized.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Processamento Concluído com Sucesso!")
print(f"Fila de risco exportada para: {OUTPUT_FILE.resolve()}")